In [1]:
%%time
import random
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import torchsummary
from datetime import datetime
import numpy as np
import pandas as pd
import xarray as xr
from sklearn.preprocessing import minmax_scale
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import gc
import os, sys
sys.path.insert(0,'/home/mingyu_seong/Works/func/python')
import mg

rs = 100
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(rs)

def minmax_scale_train_based(train_data, *other_data, eps=1e-8):
    data_min = train_data.min().item()
    data_max = train_data.max().item()
    scale = lambda x: (x - data_min) / (data_max - data_min + eps)
    return (scale(train_data),) + tuple(scale(d) for d in other_data)

def standard_scale_train_based(train_data, *other_data, eps=1e-8):
    data_mean = train_data.mean().item()
    data_std = train_data.std().item()
    scale = lambda x: (x - data_mean) / (data_std + eps)
    return (scale(train_data),) + tuple(scale(d) for d in other_data)

#=================================================================================================
# Define Model (NN)
#=================================================================================================
class NNModel(nn.Module):
    def __init__(self):
        super(NNModel, self).__init__()
        self.dense1 = nn.Linear(train_x.shape[1], 16)
        self.dense2 = nn.Linear(16, 8)
        self.output_frp = nn.Linear(8, 1)

        self.dropout = nn.Dropout(0.2)
        self.batch_norm1 = nn.BatchNorm1d(16)
        self.batch_norm2 = nn.BatchNorm1d(8)

    def forward(self, x):
        x = F.relu(self.batch_norm1(self.dense1(x)))
        x = self.dropout(x)
        x = F.relu(self.batch_norm2(self.dense2(x)))
        return self.output_frp(x)
        
#=================================================================================================
# GPU setting
#=================================================================================================
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

CPU times: user 1.46 s, sys: 223 ms, total: 1.69 s
Wall time: 1.73 s


In [ ]:
#--- Period
syr = 2001
eyr = 2024
nyr = eyr - syr + 1
tim = 'mon'

#--- Clim period
clim_syr = 2001
clim_eyr = 2020

#--- Variables to estimate monthly GPP
feature_names_ordered = [
    'T_clim','P_clim','SM_clim',               # Clims
    'T_anom','P_anom','SM_anom','SW_anom',     # Anomaly
    'SW',                                      # Saturation (Clim)
    'CO2', 'TCO2', 'SMCO2',                    # CO2 fertilization
    'TSQ']                                     # Stress

#--- Directories
idir = '../../../../_nc/input/OBS/background/r' + str(clim_syr)[2:4] + str(clim_eyr)[2:4] + '/'
idir_clim = idir + 'clim/'
idir_anom = idir + 'anom/'
idir_label= '/home/share/KCP/2025_final_input/_nc/day/MODIS/gpp/'
owdir = './wgt/'
odir = '../../../../_nc/prediction/OBS/GPP/background/'
os.makedirs(owdir, exist_ok=True) 
os.makedirs(odir,  exist_ok=True) 

#--- Input files (Climatology & Anomaly)
fin_tas = 'tas_' + tim + '_ERA5_V0_KOR_01deg_' + str(syr) + '-' + str(eyr) + '.nc'
fin_prcp  = 'pr_' + tim + '_GPM_L3_KOR_01deg_' + str(syr) + '-' + str(eyr) + '.nc'
fin_sm  = 'sfcsm_' + tim + '_ERA5_V0_KOR_01deg_' + str(syr) + '-' + str(eyr) + '.nc'
fin_rsds = 'rsds_' + tim + '_MODIS_C62_KOR_01deg_' + str(syr) + '-' + str(eyr) + '.nc'
fin_co2 =  'co2_' + tim + '_noaa_gml_KOR_01deg_' + str(syr) + '-' + str(eyr) + '.nc'

#--- Label files
fin_gpp  = 'gpp_day_MODIS_C61_KOR_01deg_20000101-20241231.nc'

data_files = {
    'T_clim': (idir_clim, fin_tas, 'clim'),
    'P_clim': (idir_clim, fin_prcp, 'clim'),
    'SM_clim': (idir_clim, fin_sm, 'clim'),
    'SW_clim': (idir_clim, fin_rsds, 'clim'),
    'CO2_clim': (idir_clim, fin_co2, 'clim'),
    'T_anom': (idir_anom, fin_tas, 'anom'),
    'P_anom': (idir_anom, fin_prcp, 'anom'),
    'SM_anom': (idir_anom, fin_sm, 'anom'),
    'SW_anom': (idir_anom, fin_rsds, 'anom'),
    'CO2_anom': (idir_anom, fin_co2, 'anom'),
}

#--- Read data
raw_data_arrays = {}
for var_name, (path, filename, da_name) in data_files.items():
    full_path = os.path.join(path, filename)
    try:
        with xr.open_dataset(full_path) as ncid:
            da = ncid[da_name].sel(time=slice(str(syr), str(eyr)))
            raw_data_arrays[var_name] = da
            
    except FileNotFoundError:
        print(f"  오류: 파일이 존재하지 않습니다: {full_path}")
        sys.exit(1)

#--- Raw time series
C0 = 1. # np.nanmean(raw_data_arrays['CO2_clim'].data)
ts_tas = raw_data_arrays['T_anom'] + raw_data_arrays['T_clim']
ts_pr  = raw_data_arrays['P_anom'] + raw_data_arrays['P_clim']
ts_sm  = raw_data_arrays['SM_anom'] + raw_data_arrays['SM_clim']
ts_sw  = raw_data_arrays['SW_anom'] + raw_data_arrays['SW_clim']
ts_co2 = raw_data_arrays['CO2_anom'] + raw_data_arrays['CO2_clim']

#--- Saturation terms
raw_data_arrays['T']   = ts_tas / ( np.nanmean(raw_data_arrays['T_clim'].data)  + ts_tas)
raw_data_arrays['P']   = ts_pr / ( np.nanmean(raw_data_arrays['P_clim'].data)  + ts_pr) # np.log(ts_pr)
raw_data_arrays['SM']  = ts_sm / ( np.nanmean(raw_data_arrays['SM_clim'].data) + ts_sm) # np.log(ts_sm)
raw_data_arrays['SW']  = ts_sw / ( np.nanmean(raw_data_arrays['SW_clim'].data) + ts_sw) # np.log(ts_sw)
raw_data_arrays['CO2'] = np.log(ts_co2/C0)

#--- Interaction terms
raw_data_arrays['TCO2'] = raw_data_arrays['T_anom'] * np.log(ts_co2/C0)
raw_data_arrays['SMCO2'] = raw_data_arrays['SM_anom'] * np.log(ts_co2/C0)
raw_data_arrays['TSM'] = raw_data_arrays['T_anom'] * raw_data_arrays['SM_anom']
raw_data_arrays['TSQ'] = raw_data_arrays['T_anom']**2

#--- input & label
features_ds = xr.Dataset()
for feature_name in feature_names_ordered:
    da = raw_data_arrays[feature_name]
    features_ds[feature_name] = da

#--- Label
ncid = xr.open_dataset(idir_label + fin_gpp)
ncid = ncid.rename({'longitude': 'lon', 'latitude': 'lat'})
weights = np.cos(np.deg2rad(ncid['lat']))
weights.name = "weights"
tmp_label = ncid['gpp'] * 1000/8                             # kg/m^2/8-day to gC/m^2/day
if tim == 'yr':
    gpp = tmp_label.sel(time=slice(str(syr),str(eyr))).weighted(weights).mean(("lat", "lon")).groupby('time.year').mean()
    gpp = gpp.rename({'year': 'time'})
elif tim == 'mon':
    gpp = tmp_label.sel(time=slice(str(syr),str(eyr))).weighted(weights).mean(("lat", "lon")).resample(time='1ME').mean()
ncid.close() ; del ncid
label_da = gpp

#--- Add Month of Year
if tim == 'mon':
    moy = []
    for imon in np.arange(12):
        mon = imon + 1
        moy.append(float(np.sin(2*np.pi/12*mon)))
    features_ds['MOY'] = xr.DataArray(moy*nyr,
                            dims=['time'],
                            coords={'time': xr.open_dataset(idir_clim+fin_tas).time.data})
    feature_names_ordered = feature_names_ordered + ['MOY']

#--- Combined inputs
combined_features_da = features_ds.to_array(dim='variable')

In [3]:
%%time
for ifold in np.arange(12):
    # ifold = 11
    train_start  = None ; train_end  = None
    train_start0 = None ; train_end0 = None
    train_start1 = None ; train_end1 = None
    valid_start  = None ; valid_end  = None
    valid_start0 = None ; valid_end0 = None
    valid_start1 = None ; valid_end1 = None
    test_start   = None ; test_end   = None
    
    # Test: 17-yr / Validation: 5-yr / Test: 2-yr   (r: 0.83 / Reg. Coef: 0.93)
    if ifold == 0:
        train_start = '2003' ; train_end = '2019'
        valid_start = '2020' ; valid_end = '2024'
        test_start  = '2001' ; test_end  = '2002'
    elif ifold == 1:
        train_start  = '2005' ; train_end  = '2021'
        valid_start0 = '2022' ; valid_end0 = '2024'
        valid_start1 = '2001' ; valid_end1 = '2002'
        test_start   = '2003' ; test_end   = '2004'
    elif ifold == 2:   
        train_start  = '2005' ; train_end  = '2023'
        valid_start0 = '2024' ; valid_end0 = '2024'
        valid_start1 = '2001' ; valid_end1 = '2004'
        test_start   = '2005' ; test_end   = '2006'
    elif ifold == 3:   
        train_start0 = '2001' ; train_end0 = '2001'
        train_start1 = '2009' ; train_end1 = '2024'
        valid_start  = '2002' ; valid_end  = '2006'
        test_start   = '2007' ; test_end   = '2008'
    elif ifold == 4:   
        train_start0 = '2001' ; train_end0 = '2003'
        train_start1 = '2011' ; train_end1 = '2024'
        valid_start  = '2004' ; valid_end  = '2008'
        test_start   = '2009' ; test_end   = '2010'
    elif ifold == 5:   
        train_start0 = '2001' ; train_end0 = '2005'
        train_start1 = '2013' ; train_end1 = '2024'
        valid_start  = '2006' ; valid_end  = '2010'
        test_start   = '2011' ; test_end   = '2012'
    elif ifold == 6:   
        train_start0 = '2001' ; train_end0 = '2007'
        train_start1 = '2015' ; train_end1 = '2024'
        valid_start  = '2008' ; valid_end  = '2012'
        test_start   = '2013' ; test_end   = '2014'
    elif ifold == 7:   
        train_start0 = '2001' ; train_end0 = '2009'
        train_start1 = '2017' ; train_end1 = '2024'
        valid_start  = '2010' ; valid_end  = '2014'
        test_start   = '2015' ; test_end   = '2016'
    elif ifold == 8:   
        train_start0 = '2001' ; train_end0 = '2011'
        train_start1 = '2019' ; train_end1 = '2024'
        valid_start  = '2012' ; valid_end  = '2016'
        test_start   = '2017' ; test_end   = '2018'
    elif ifold == 9:   
        train_start0 = '2001' ; train_end0 = '2013'
        train_start1 = '2021' ; train_end1 = '2024'
        valid_start  = '2014' ; valid_end  = '2018'
        test_start   = '2019' ; test_end   = '2020'
    elif ifold == 10:   
        train_start0 = '2001' ; train_end0 = '2015'
        train_start1 = '2023' ; train_end1 = '2024'
        valid_start  = '2016' ; valid_end  = '2020'
        test_start   = '2021' ; test_end   = '2022'
    elif ifold == 11:   
        train_start = '2001' ; train_end = '2017'
        valid_start = '2018' ; valid_end = '2022'
        test_start  = '2023' ; test_end  = '2024'
   
    #--- Weight output
    wfou = 'GPP_wgt_' + tim + '_fold' + str(ifold).zfill(2) + '_test_period_' + test_start + '-' + test_end + '.pth'
    
    #--- Train
    if train_start0 is not None and train_start1 is not None:
        # 1) First period
        train_features_da_part0 = combined_features_da.sel(time=slice(train_start0, train_end0))
        train_label_da_part0 = label_da.sel(time=slice(train_start0, train_end0))
        # 2) Second period
        train_features_da_part1 = combined_features_da.sel(time=slice(train_start1, train_end1))
        train_label_da_part1 = label_da.sel(time=slice(train_start1, train_end1))
        # 3) Merge periods
        train_features_da = xr.concat([train_features_da_part0, train_features_da_part1], dim='time')
        train_label_da = xr.concat([train_label_da_part0, train_label_da_part1], dim='time')
    else:
        train_features_da = combined_features_da.sel(time=slice(train_start, train_end))
        train_label_da = label_da.sel(time=slice(train_start, train_end))

    #--- Validation
    if valid_start0 is not None and valid_start1 is not None:
        # 1) First period
        valid_features_da_part0 = combined_features_da.sel(time=slice(valid_start0, valid_end0))
        valid_label_da_part0 = label_da.sel(time=slice(valid_start0, valid_end0))
        # 2) Second period
        valid_features_da_part1 = combined_features_da.sel(time=slice(valid_start1, valid_end1))
        valid_label_da_part1 = label_da.sel(time=slice(valid_start1, valid_end1))
        # 3) Merge periods
        valid_features_da = xr.concat([valid_features_da_part0, valid_features_da_part1], dim='time')
        valid_label_da = xr.concat([valid_label_da_part0, valid_label_da_part1], dim='time')
    else:
        valid_features_da = combined_features_da.sel(time=slice(valid_start, valid_end))
        valid_label_da = label_da.sel(time=slice(valid_start, valid_end))

    #--- Test
    test_features_da = combined_features_da.sel(time=slice(test_start, test_end))      
    test_label_da = label_da.sel(time=slice(test_start, test_end))

    stacked_train_features_list, stacked_valid_features_list, stacked_test_features_list = [],[],[]
    for i, feature_name in enumerate(feature_names_ordered):
        stacked_train_feature_array = train_features_da[i].values
        stacked_valid_feature_array  = valid_features_da[i].values
        stacked_test_feature_array  = test_features_da[i].values
        stacked_train_features_list.append(stacked_train_feature_array)
        stacked_valid_features_list.append(stacked_valid_feature_array)
        stacked_test_features_list.append(stacked_test_feature_array)
    
    #--- Separate data (train / validation / test)
    train_x = np.transpose(np.vstack(stacked_train_features_list))
    valid_x = np.transpose(np.vstack(stacked_valid_features_list))
    test_x  = np.transpose(np.vstack(stacked_test_features_list))
    train_y = train_label_da.values
    valid_y = valid_label_da.values
    test_y  = test_label_da.values

    #--- Remove NaN
    nan_x_rows = np.isnan(train_x).any(axis=1)
    nan_y_rows = np.isnan(train_y)
    nan_rows = nan_x_rows | nan_y_rows
    train_x = train_x[~nan_rows]
    train_y = train_y[~nan_rows]

    nan_x_rows = np.isnan(valid_x).any(axis=1)
    nan_y_rows = np.isnan(valid_y)
    nan_rows = nan_x_rows | nan_y_rows
    valid_x = valid_x[~nan_rows]
    valid_y = valid_y[~nan_rows]

    #--- Normalize variables
    for i in np.arange(train_x.shape[1]):
        train_x[:,i], valid_x[:,i], test_x[:,i] = standard_scale_train_based(train_x[:,i], valid_x[:,i], test_x[:,i])

    # Convert numpy arrays to PyTorch tensors
    train_x = torch.tensor(train_x, dtype=torch.float32, device=device)
    train_y = torch.tensor(train_y, dtype=torch.float32, device=device)
    valid_x = torch.tensor(valid_x, dtype=torch.float32, device=device)
    valid_y = torch.tensor(valid_y, dtype=torch.float32, device=device)
    test_x = torch.tensor(test_x, dtype=torch.float32, device=device)
    test_y = torch.tensor(test_y, dtype=torch.float32, device=device)
    
    # Dataset and DataLoader for batching
    train_data = TensorDataset(train_x, train_y)
    valid_data = TensorDataset(valid_x, valid_y)
    test_data = TensorDataset(test_x, test_y)
    
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
    valid_loader = DataLoader(valid_data, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

    #=================================================================================================
    # Model Training
    #=================================================================================================
    model = NNModel().to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    EPOCHS = 1000
    patience = 20
    best_val_loss = float('inf')
    epochs_no_improve = 0
    
    train_losses, val_losses = [],[]
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs.squeeze(), batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
    
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_x, batch_y in valid_loader:
                outputs = model(batch_x)
                loss = criterion(outputs.squeeze(), batch_y)
                val_loss += loss.item()
    
        train_losses.append(train_loss/len(train_loader))
        val_losses.append(val_loss/len(valid_loader))

        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), owdir + wfou)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping!", epoch)
                break

    #--- Inference
    model.eval()
    y_pred = []
    with torch.no_grad():
        for batch_x, _ in test_loader:
            outputs = model(batch_x)
            y_pred.extend(outputs.squeeze().cpu().numpy())
    pred = np.array(y_pred)

    #--- Save
    pred_data = xr.DataArray(pred,
                            dims=['time'],
                            coords={'time': test_label_da.time.data},
                            attrs={'long_name': 'Predicted GPP',
                                   'units': 'gC/m^2/day',}) # '_FillValue': 1e+30})
    
    raw_data = xr.DataArray(test_label_da,
                            dims=['time'],
                            coords={'time': test_label_da.time.data},
                            attrs={'long_name': 'Predicted GPP',
                                   'units': 'gC/m^2/day',}) # '_FillValue': 1e+30})

    ds = xr.Dataset()
    ds['pred'] = pred_data
    ds['raw'] = raw_data
    ds.to_netcdf(odir + 'predicted_GPP_'+ tim +'_FFNN_' + \
                 test_start.replace('-','') + '-' +                    \
                 test_end.replace('-','') + '_seed' + str(rs) + '.nc')
    del train_x, train_y, test_x, test_y
    del pred_data, raw_data, ds

#--- Merging times
fou = 'predicted_GPP_' + tim + '_FFNN_' + str(syr) + '-' +  str(eyr) + \
      '_seed' + str(rs) + '.nc'
ftmp = mg.get_files(pin=odir,fin='predicted_GPP_'+ tim +'_FFNN_*.nc')
flist = [os.path.join(odir, filename ) for filename  in ftmp]
mg.cdo('mergetime',input=flist, output=odir+fou, options='-s -L -O')

sys.exit()

Early stopping! 164
Early stopping! 252
Early stopping! 94
Early stopping! 121
Early stopping! 156
Early stopping! 242
Early stopping! 198
Early stopping! 146
Early stopping! 204
Early stopping! 168
Early stopping! 212
Early stopping! 142


SystemExit: 